# Notebook 08: Background counterfactual (E12, hypothesis H2)

**Question.** Does the Phase 1 model read the background rather than the leaf? Ghanaian (CCMT) leaves are cut out of their plain backgrounds and shown to the frozen Phase 1 model (EfficientNet-B0, seed 42, FP32 weights of the deployed instance) six ways: the original photo; the **background only** (leaf region filled with its own mean colour); the same leaf on a **white** background, a **dark** background, a **Ugandan field** background taken from a D1 training image, and a **random non-plant texture**. A prediction that changes when only the background changes is direct evidence of a shortcut.

**Data.** The `adapt` halves of CCMT (`d3c_adapt`: C01 CBB, C02 CMD, C03 healthy; `d3m_adapt`: M01 MSV, M05 healthy), capped at 400 images per class. The test halves are left for scoring only. Nothing is trained or tuned here, and the Phase 1 model is not changed. **Exploratory** (D3 has been seen), but causal in design, as registered.

**Endpoints (registered).** (1) Share of leaves whose predicted class changes when only the background changes, per condition. (2) Recall of CBB and CMD (and MSV) on the Ugandan-field background versus the plain original. (3) Accuracy from the background alone versus the majority-class rate. Confidence and abstention are reported per condition. Images whose mask fails a quality check are excluded and counted.

**Runtime.** Any GPU (T4 is enough; A100 is fine). Segmentation runs on the CPU (about 15–25 minutes), prediction on the GPU (a few minutes). Resumable.

**Version 3 (5 Nov 2026).** The first run showed that CCMT is a mix of leaves on paper or card and field photographs, and that leaves often fill the frame, so the border-colour segmentation rejected most images. The second version seeded GrabCut from saturation alone and let field photos through with pale leaf regions treated as paper. This version first classes every photo as **paper** or **field** (unsaturated, non-green pixels covering at least 15% of the image and reaching its border), segments only paper photos, and requires the removed region to be almost entirely paper-like. It also reports, per class, how many CCMT photos are paper and how many are field, which is itself evidence about label-correlated backgrounds. Results go to `records/nb08_v3`; earlier records are kept, and the change is written to the deviations file.

**What to check while it runs.** Cell 4 prints the segmentation counts and shows two sheets per crop (good masks and failed images). Cell 5 shows one sheet per crop of six leaves under the six conditions. Cell 6 prints the endpoint tables and downloads `nb08_results.zip`.

In [ ]:
# 1. Setup, integrity checks (registration text, frozen Phase 1 choices), environment record
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, time, glob, tarfile, hashlib, datetime, platform, subprocess, gc, math, collections, concurrent.futures
import numpy as np, pandas as pd
import tensorflow as tf, keras
DRIVE = '/content/drive/MyDrive/ocdf'; WORK = '/content/working'
OUT = f'{DRIVE}/records/nb08_v3'; MASKS = '/content/masks'; SHEETS = f'{OUT}/sheets'      # v3: segmentation method revised after two runs (see cell 4)
for d in [OUT, MASKS, SHEETS, WORK]: os.makedirs(d, exist_ok=True)
txt_path = f'{DRIVE}/records/protocol_v3_registration_text.md'
assert hashlib.sha256(open(txt_path, 'rb').read()).hexdigest() == open(txt_path + '.sha256').read().strip(), 'registration text changed'
reg = json.load(open(f'{DRIVE}/records/protocol_v3_registration.json')); print('registration', reg['registered_utc'])
fpath = f'{DRIVE}/records/frozen_choices_v2.json'
assert hashlib.sha256(open(fpath, 'rb').read()).hexdigest() == open(f'{DRIVE}/records/frozen_choices_v2.sha256').read().strip(), 'frozen choices changed'
F = json.load(open(fpath)); DEP = F['deployed_instance']
for crop, d in DEP.items(): print(crop, 'deployed:', d['architecture'], 'seed', d['seed'], d['format'], 'T =', round(d['temperature_T'], 4), 'threshold', d['threshold'])
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
gpu_name = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() if gpus else 'none'
env = {'notebook': '08_background_counterfactual', 'run_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'python': sys.version.split()[0],
       'tensorflow': tf.__version__, 'keras': keras.__version__, 'numpy': np.__version__, 'gpu': gpu_name, 'cpu_count': os.cpu_count(), 'platform': platform.platform()}
json.dump(env, open(f'{OUT}/environment_nb08.json', 'w'), indent=1); print(json.dumps(env, indent=1))

In [ ]:
# 2. Manifest v3, image selection (adapt halves, capped), Ugandan field backgrounds from D1 training images
seal = json.load(open(f'{DRIVE}/records/phase2_data_v3.json'))
mpath = f'{DRIVE}/manifests/{seal["manifest_v3"]}'
assert hashlib.sha256(open(mpath, 'rb').read()).hexdigest() == seal['sha256'], 'manifest_v3 changed since it was frozen'
M = pd.read_csv(mpath, dtype=str)
M = M[M.partition != 'd5_test'].copy()                      # SEALED: never loaded
for src in ['D1', 'D3']:
    if not os.path.isdir(f'{WORK}/{src}'):
        with tarfile.open(f'{DRIVE}/working/working_{src}.tar') as tar: tar.extractall(WORK, filter='data')
CLASSES = {'cassava': ['C01', 'C02', 'C03'], 'maize': ['M01', 'M05']}       # maize: the two classes CCMT shares with the Phase 1 label set
N_OUT = {'cassava': 3, 'maize': 5}; LABEL_IDX = {'cassava': {'C01': 0, 'C02': 1, 'C03': 2}, 'maize': {'M01': 0, 'M05': 4}}
CAP = 400; SEED = 2026
rng = np.random.default_rng(SEED)
sel = []
for crop, part in [('cassava', 'd3c_adapt'), ('maize', 'd3m_adapt')]:
    d = M[(M.partition == part) & (M.research_label.isin(CLASSES[crop]))]
    for lab, g in d.groupby('research_label'):
        g = g.sort_values('sample_id')
        take = g if len(g) <= CAP else g.iloc[np.sort(rng.choice(len(g), CAP, replace=False))]
        sel.append(take.assign(crop_eval=crop))
S = pd.concat(sel).reset_index(drop=True)
print('selected CCMT images:'); print(S.groupby(['crop_eval', 'research_label']).size())
# background pool: D1 training images (Ugandan field), stratified over the three cassava classes so the background label is known
D1 = M[(M.source_id == 'D1') & (M.partition == 'train')].sort_values('sample_id')
BG = pd.concat([g.iloc[np.sort(rng.choice(len(g), min(150, len(g)), replace=False))] for _, g in D1.groupby('research_label')]).reset_index(drop=True)
print('D1 field-background pool:', BG.research_label.value_counts().to_dict())

In [ ]:
# 3. Leaf segmentation (paper-or-field classing, saturation/hue prior + GrabCut; version 3), quality checks, and the compositing operations
import cv2
cv2.setNumThreads(1)
SEG = {'version': 3, 'grabcut_iters': 5, 'leaf_fraction_min': 0.04, 'leaf_fraction_max': 0.85, 'feather_px': 3, 'min_component_share': 0.10,
       'paper': 'HSV (OpenCV scale): S < 40 and V > 110, or S < 90 and V > 110 with hue outside the plant band 25-95', 'sure_fg': 'S > 110 and V > 40',
       'paper_present_min': 0.15, 'paper_border_min': 0.05, 'background_paper_min': 0.85}

def read_rgb(path):
    im = cv2.imread(path, cv2.IMREAD_COLOR); assert im is not None, path
    return cv2.cvtColor(im, cv2.COLOR_BGR2RGB)

def paper_mask(rgb):
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV); H, S, V = (hsv[..., i].astype(np.int32) for i in range(3))
    plant_hue = (H >= 25) & (H <= 95)                                   # yellow-green-green: leaf tissue, including pale or sunlit parts
    paper = ((S < 40) & (V > 110)) | ((S < 90) & (V > 110) & ~plant_hue)  # white, grey, blue-white or beige paper and card, with shadows
    return paper, S, V

def background_type(rgb):
    """'paper' when unsaturated paper-like pixels cover >= 15% of the image and reach its border, else 'field'."""
    paper, S, V = paper_mask(rgb); b = 8
    frame = np.concatenate([paper[:b].ravel(), paper[-b:].ravel(), paper[:, :b].ravel(), paper[:, -b:].ravel()])
    ok = paper.mean() >= SEG['paper_present_min'] and frame.mean() >= SEG['paper_border_min']
    return ('paper' if ok else 'field'), float(paper.mean()), float(frame.mean())

def segment_leaf(rgb):
    """Returns (mask uint8 0/255, info dict). CCMT mixes leaves laid on paper or card with field photographs. Only paper-background
    photos can be segmented reliably without labels, so a photo is first classed as 'paper' or 'field' (background_type). For paper
    photos, paper-like pixels seed GrabCut as background and saturated pixels as foreground; the leaf may touch any border. After
    GrabCut the removed region must be almost entirely paper-like, and the leaf fraction plausible; otherwise the photo is excluded."""
    h, w = rgb.shape[:2]
    btype, pf, bf = background_type(rgb)
    info = {'background_type': btype, 'paper_fraction_image': round(pf, 4), 'paper_fraction_border': round(bf, 4)}
    if btype != 'paper':
        return None, dict(info, status='background_not_plain')
    paper, S, V = paper_mask(rgb)
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    sure_bg = cv2.erode(((S < 40) & (V > 110)).astype(np.uint8), k5).astype(bool)
    sure_fg = cv2.erode(((S > 110) & (V > 40)).astype(np.uint8), k5).astype(bool)
    gc = np.where(paper, cv2.GC_PR_BGD, cv2.GC_PR_FGD).astype(np.uint8)
    gc[sure_bg] = cv2.GC_BGD; gc[sure_fg] = cv2.GC_FGD
    if ((gc == cv2.GC_PR_FGD) | (gc == cv2.GC_FGD)).mean() < SEG['leaf_fraction_min']:
        return None, dict(info, status='no_foreground_seed')
    bgd = np.zeros((1, 65), np.float64); fgd = np.zeros((1, 65), np.float64)
    try:
        cv2.grabCut(cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR), gc, None, bgd, fgd, SEG['grabcut_iters'], cv2.GC_INIT_WITH_MASK)
    except cv2.error:
        return None, dict(info, status='grabcut_error')
    mask = ((gc == cv2.GC_FGD) | (gc == cv2.GC_PR_FGD)).astype(np.uint8)
    n, lab_cc, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if n <= 1:
        return None, dict(info, status='empty_mask')
    areas = stats[1:, cv2.CC_STAT_AREA]; keep = np.where(areas >= SEG['min_component_share'] * areas.max())[0] + 1
    mask = np.isin(lab_cc, keep).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
    holes = np.zeros((h + 2, w + 2), np.uint8); flood = mask.copy(); cv2.floodFill(flood, holes, (0, 0), 1)
    if mask[0, 0] == 0: mask = (mask | (1 - flood)).astype(np.uint8)      # fill holes when the corner is background
    frac = float(mask.mean()); info['leaf_fraction'] = round(frac, 4)
    bg_paper = float(paper[mask == 0].mean()) if (mask == 0).any() else 0.0; info['background_paper_fraction'] = round(bg_paper, 4)
    if frac < SEG['leaf_fraction_min'] or frac > SEG['leaf_fraction_max']:
        return None, dict(info, status='leaf_fraction_out_of_range')
    if bg_paper < SEG['background_paper_min']:
        return None, dict(info, status='mask_background_not_paper')
    ys, xs = np.where(mask > 0); info['bbox'] = [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())]
    return mask * 255, dict(info, status='ok')

def feathered_alpha(mask):
    m = cv2.erode(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))       # drop the 2 px rim that may hold background colour
    a = cv2.GaussianBlur(m.astype(np.float32) / 255.0, (0, 0), SEG['feather_px'])
    return np.clip(a, 0, 1)[..., None]

def composite(leaf_rgb, alpha, background_rgb):
    return np.clip(leaf_rgb * alpha + background_rgb * (1 - alpha), 0, 255).astype(np.uint8)

def background_only(rgb, mask):
    out = rgb.copy(); m = mask > 0
    out[m] = rgb[m].reshape(-1, 3).mean(0).round().astype(np.uint8)        # leaf region replaced by its own mean colour
    return out

def plain_background(shape, value):
    return np.full(shape, value, np.uint8)

def field_background(d1_rgb, shape):
    """A Ugandan field photograph resized to cover the Ghanaian image (centre crop keeps the aspect ratio)."""
    h, w = shape[:2]; H, W = d1_rgb.shape[:2]; s = max(h / H, w / W)
    r = cv2.resize(d1_rgb, (max(w, int(round(W * s))), max(h, int(round(H * s)))), interpolation=cv2.INTER_AREA)
    y0 = (r.shape[0] - h) // 2; x0 = (r.shape[1] - w) // 2
    return r[y0:y0 + h, x0:x0 + w]

def texture_background(shape, rng):
    """A procedural non-plant texture: blurred noise, stripes, checks or a gradient, in non-green colours."""
    h, w = shape[:2]
    def colour():
        hue = rng.choice([rng.uniform(0, 40), rng.uniform(170, 180)])                 # OpenCV hue 0-180; avoids 40-170 (greens/cyans)
        hsv = np.array([[[hue, rng.uniform(60, 200), rng.uniform(80, 230)]]], np.uint8)
        return cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)[0, 0].astype(np.float32)
    c1, c2 = colour(), colour(); kind = rng.choice(['noise', 'stripes', 'checks', 'gradient'])
    if kind == 'noise':
        t = rng.random((h, w), np.float32); t = cv2.GaussianBlur(t, (0, 0), rng.uniform(2, 12)); t = (t - t.min()) / (t.max() - t.min() + 1e-6)
    elif kind == 'stripes':
        period = rng.uniform(10, 60); ang = rng.uniform(0, np.pi); yy, xx = np.mgrid[0:h, 0:w]
        t = 0.5 + 0.5 * np.sin(2 * np.pi * (xx * np.cos(ang) + yy * np.sin(ang)) / period).astype(np.float32)
    elif kind == 'checks':
        size = int(rng.uniform(12, 50)); yy, xx = np.mgrid[0:h, 0:w]; t = (((yy // size) + (xx // size)) % 2).astype(np.float32)
    else:
        yy, xx = np.mgrid[0:h, 0:w]; ang = rng.uniform(0, np.pi); t = (xx * np.cos(ang) + yy * np.sin(ang)); t = ((t - t.min()) / (t.max() - t.min() + 1e-6)).astype(np.float32)
    img = c1 * t[..., None] + c2 * (1 - t[..., None])
    img += rng.normal(0, 4, img.shape)
    return np.clip(img, 0, 255).astype(np.uint8), kind

CONDITIONS = ['original', 'background_only', 'white', 'dark', 'field', 'texture']

def make_conditions(rgb, mask, d1_rgb, rng):
    """All six images for one leaf. Returns dict condition -> RGB uint8, plus the texture kind."""
    a = feathered_alpha(mask); shape = rgb.shape
    tex, kind = texture_background(shape, rng)
    return {'original': rgb, 'background_only': background_only(rgb, mask),
            'white': composite(rgb, a, plain_background(shape, 245)), 'dark': composite(rgb, a, plain_background(shape, 20)),
            'field': composite(rgb, a, field_background(d1_rgb, shape)), 'texture': composite(rgb, a, tex)}, kind

def _segment_job(args):
    sid, path = args
    try:
        rgb = read_rgb(path); mask, info = segment_leaf(rgb)
        if mask is not None: cv2.imwrite(f'{MASKS}/{sid}.png', mask)
        return dict(info, sample_id=sid)
    except Exception as e:
        return {'sample_id': sid, 'status': f'error: {type(e).__name__}'}

In [ ]:
# 4. Segment every selected image (parallel, resumable), count failures, show sheets
seg_path = f'{OUT}/e12_segmentation.csv'
done = pd.read_csv(seg_path, dtype=str) if os.path.exists(seg_path) else pd.DataFrame(columns=['sample_id'])
todo = [(r.sample_id, f'{WORK}/{r.working_path}') for r in S.itertuples() if r.sample_id not in set(done.sample_id) or not os.path.exists(f'{MASKS}/{r.sample_id}.png')]
print(len(todo), 'images to segment (', len(S) - len(todo), 'already done )')
rows = [] if len(todo) == len(S) else done.to_dict('records')
t0 = time.time()
with concurrent.futures.ProcessPoolExecutor(max_workers=max(1, os.cpu_count())) as ex:
    for i, r in enumerate(ex.map(_segment_job, todo, chunksize=8)):
        rows.append(r)
        if (i + 1) % 200 == 0: print(f'  {i + 1}/{len(todo)}  {(time.time() - t0) / 60:.1f} min')
SEGR = pd.DataFrame(rows).drop_duplicates('sample_id', keep='last'); SEGR.to_csv(seg_path, index=False)
S2 = S.merge(SEGR[['sample_id', 'status', 'background_type', 'leaf_fraction', 'background_paper_fraction']], on='sample_id', how='left')
print('\nbackground type by crop and class (paper = leaf laid on paper or card; field = photographed in situ):'); print(pd.crosstab([S2.crop_eval, S2.research_label], S2.background_type))
print('\nsegmentation status by crop and class:'); print(pd.crosstab([S2.crop_eval, S2.research_label], S2.status))
OK = S2[S2.status == 'ok'].reset_index(drop=True)
seg_summary = {'selected': int(len(S2)), 'ok': int(len(OK)), 'failed': int((S2.status != 'ok').sum()),
               'failed_by_status': S2[S2.status != 'ok'].status.value_counts().to_dict(),
               'by_crop_class': {f'{c}/{l}': {'ok': int((g.status == 'ok').sum()), 'failed': int((g.status != 'ok').sum())} for (c, l), g in S2.groupby(['crop_eval', 'research_label'])},
               'background_type_by_class': {f'{c}/{l}': {'n': int(len(g)), 'paper': int((g.background_type == 'paper').sum()), 'field': int((g.background_type == 'field').sum()), 'paper_share': round(float((g.background_type == 'paper').mean()), 3)} for (c, l), g in S2.groupby(['crop_eval', 'research_label'])},
               'settings': SEG}
json.dump(seg_summary, open(f'{OUT}/e12_segmentation_summary.json', 'w'), indent=1, default=str)
# sheets: good masks (overlay) and failures, per crop
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
def tile(img, size=200):
    h, w = img.shape[:2]; s = size / max(h, w); return cv2.resize(img, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA)
for crop in CLASSES:
    good = OK[OK.crop_eval == crop]; bad = S2[(S2.crop_eval == crop) & (S2.status != 'ok')]
    ex_g = good.iloc[np.sort(rng.choice(len(good), min(24, len(good)), replace=False))] if len(good) else good
    fig, axes = plt.subplots(4, 6, figsize=(15, 10.5)); axes = axes.ravel()
    for ax in axes: ax.axis('off')
    for ax, r in zip(axes, ex_g.itertuples()):
        rgb = read_rgb(f'{WORK}/{r.working_path}'); m = cv2.imread(f'{MASKS}/{r.sample_id}.png', 0)
        ov = rgb.copy(); ov[m == 0] = (0.35 * ov[m == 0] + 0.65 * np.array([255, 0, 255])).astype(np.uint8)
        ax.imshow(tile(ov)); ax.set_title(f'{r.sample_id} {r.research_label} leaf {float(r.leaf_fraction):.2f}', fontsize=7)
    fig.suptitle(f'{crop}: GrabCut masks on {len(ex_g)} random CCMT images (magenta = background removed)'); plt.tight_layout()
    fig.savefig(f'{SHEETS}/mask_sheet_{crop}.png', dpi=110); plt.show(); plt.close(fig)
    if len(bad):
        ex_b = bad.iloc[:24]; fig, axes = plt.subplots(4, 6, figsize=(15, 10.5)); axes = axes.ravel()
        for ax in axes: ax.axis('off')
        for ax, r in zip(axes, ex_b.itertuples()):
            ax.imshow(tile(read_rgb(f'{WORK}/{r.working_path}'))); ax.set_title(f'{r.sample_id} {r.research_label}\n{r.status}', fontsize=7)
        fig.suptitle(f'{crop}: {len(bad)} images excluded (mask failed the quality check); first {len(ex_b)} shown'); plt.tight_layout()
        fig.savefig(f'{SHEETS}/failed_sheet_{crop}.png', dpi=110); plt.show(); plt.close(fig)
print(json.dumps({k: v for k, v in seg_summary.items() if k != 'settings'}, indent=1, default=str))
# Deviation record: the registration assumed plain studio backgrounds for all CCMT images. The first run (border-colour method, records/nb08)
# found that many CCMT photos are field photos or leaves filling the frame. H2 therefore uses the CCMT images photographed on paper or card;
# field photos are excluded and counted here. The experiment's design and endpoints are unchanged.
dev_path = f'{DRIVE}/records/protocol_v3_deviations.json'
devs = json.load(open(dev_path)) if os.path.exists(dev_path) else []
if not any(d.get('notebook') == '08' for d in devs):
    devs.append({'date_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'notebook': '08',
                 'what': 'The registration described CCMT backgrounds as plain studio backgrounds. Inspection showed a mix of leaves on paper or card and field photographs, '
                         'and leaves that fill the frame. The first segmentation method (border colour, records/nb08) rejected 1,409 of 1,702 images; the second (saturation prior, '
                         'records/nb08_v2) let field photos through with pale leaf regions treated as paper. The third (records/nb08_v3) first classes each photo as paper or field '
                         'from unsaturated, non-green pixels reaching the image border, segments only paper photos, and requires the removed region to be paper-like. '
                         'Each revision was made from the mask sheets, before any H2 endpoint from that run was read; the superseded records are kept.',
                 'remedy': f'H2 is run on the CCMT images whose background is paper or card: {seg_summary["ok"]} of {seg_summary["selected"]} selected images '
                           f'({json.dumps(seg_summary["failed_by_status"])} excluded). Excluded images are counted per class in e12_segmentation_summary.json. '
                           'Design, conditions and endpoints are as registered; the H2 result applies to paper-background images and is reported as such.',
                 'affects_results': True})
    json.dump(devs, open(dev_path, 'w'), indent=1); print('deviation recorded for Notebook 08')

In [ ]:
# 5. Build the six conditions per leaf and score them with the frozen Phase 1 model (resumable per crop)
IMG = 224
def prep(batch_rgb):
    x = tf.stack([tf.image.resize_with_pad(tf.convert_to_tensor(im), IMG, IMG, method='bilinear') for im in batch_rgb])   # images differ in size
    return tf.cast(tf.clip_by_value(tf.round(x), 0, 255), tf.float32)
def temperature(p, T):
    z = np.log(np.clip(p, 1e-12, 1)) / T; z -= z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)
pred_path = f'{OUT}/e12_predictions.csv'
PRED = pd.read_csv(pred_path) if os.path.exists(pred_path) else pd.DataFrame()
for crop in CLASSES:
    if len(PRED) and (PRED.crop == crop).any(): print(crop, 'already scored'); continue
    dep = DEP[crop]; assert dep['architecture'] == 'EfficientNetB0' and int(dep['seed']) == 42, dep
    model = keras.models.load_model(f'{DRIVE}/runs/{crop}_{dep["architecture"]}_s{dep["seed"]}/best_model.keras')
    T, thr = float(dep['temperature_T']), float(dep['threshold'])
    d = OK[OK.crop_eval == crop].reset_index(drop=True); crng = np.random.default_rng(SEED + len(crop))
    bg_pick = crng.integers(0, len(BG), len(d))
    rows = []; batch = []; meta = []; sheet = []
    def flush():
        if not batch: return
        p = model.predict(prep(batch), batch_size=64, verbose=0); pT = temperature(p, T)
        for m, pr in zip(meta, pT):
            rows.append(dict(m, pred=int(pr.argmax()), conf=float(pr.max()), accept=bool(pr.max() >= thr), **{f'p{i}': float(v) for i, v in enumerate(pr)}))
        batch.clear(); meta.clear()
    t0 = time.time()
    for i, r in enumerate(d.itertuples()):
        rgb = read_rgb(f'{WORK}/{r.working_path}'); mask = cv2.imread(f'{MASKS}/{r.sample_id}.png', 0)
        bgr = BG.iloc[bg_pick[i]]; d1 = read_rgb(f'{WORK}/{bgr.working_path}')
        conds, kind = make_conditions(rgb, mask, d1, crng)
        if len(sheet) < 6: sheet.append((r.sample_id, r.research_label, conds))
        for c in CONDITIONS:
            batch.append(conds[c]); meta.append({'crop': crop, 'sample_id': r.sample_id, 'group_id': r.group_id, 'label': r.research_label, 'y': LABEL_IDX[crop][r.research_label],
                                                 'condition': c, 'bg_sample_id': bgr.sample_id if c == 'field' else '', 'bg_label': bgr.research_label if c == 'field' else '',
                                                 'texture_kind': kind if c == 'texture' else ''})
        if len(batch) >= 192: flush()
        if (i + 1) % 200 == 0: print(f'  {crop} {i + 1}/{len(d)}  {(time.time() - t0) / 60:.1f} min')
    flush()
    PRED = pd.concat([PRED, pd.DataFrame(rows)], ignore_index=True); PRED.to_csv(pred_path, index=False)
    print(crop, 'scored', len(d), 'leaves x', len(CONDITIONS), 'conditions in', round((time.time() - t0) / 60, 1), 'min')
    fig, axes = plt.subplots(len(sheet), len(CONDITIONS), figsize=(2.6 * len(CONDITIONS), 2.4 * len(sheet)))
    for a, (sid, lab, conds) in enumerate(sheet):
        for b, c in enumerate(CONDITIONS):
            ax = axes[a, b]; ax.imshow(tile(conds[c])); ax.set_xticks([]); ax.set_yticks([])
            pr = PRED[(PRED.sample_id == sid) & (PRED.condition == c)].iloc[0]
            inv = {v: k for k, v in LABEL_IDX[crop].items()}; pl = inv.get(int(pr.pred), f'class{int(pr.pred)}')
            ax.set_title(f'{c}\npred {pl} ({pr.conf:.2f}){"" if pr.accept else " abstain"}', fontsize=8, color='black' if pl == lab else 'firebrick')
            if b == 0: ax.set_ylabel(f'{sid}\ntrue {lab}', fontsize=8)
    fig.suptitle(f'{crop}: the same leaf under six backgrounds, Phase 1 model predictions (red = wrong)'); plt.tight_layout()
    fig.savefig(f'{SHEETS}/conditions_sheet_{crop}.png', dpi=110); plt.show(); plt.close(fig)
    del model; gc.collect()
print(PRED.groupby(['crop', 'condition']).size().unstack())

In [ ]:
# ---- OCDF F6 evaluation helpers (Notebook 05): metrics, abstention, group bootstrap ----
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, precision_recall_fscore_support, confusion_matrix

def full_metrics(p, y, eval_labels, n_out, threshold=None, groups=None):
    """Evaluation metrics (docs/METHODS.md). `eval_labels`: class indices present in this evaluation set (others are 'not evaluated').
    Predictions keep all n_out outputs (never masked), so predicting an absent class counts as an error."""
    pred = p.argmax(1); L = list(eval_labels)
    pr, rc, f1, sup = precision_recall_fscore_support(y, pred, labels=L, zero_division=0)
    out = {'n_images': int(len(y)), 'n_groups': int(len(set(groups))) if groups is not None else None,
           'accuracy': float(accuracy_score(y, pred)),
           'balanced_accuracy': float(np.mean([np.mean(pred[y == c] == c) for c in L])),
           'macro_f1': float(f1_score(y, pred, labels=L, average='macro', zero_division=0)),
           'per_class': {int(c): {'precision': float(a), 'recall': float(b), 'f1': float(d), 'support': int(s)}
                         for c, a, b, d, s in zip(L, pr, rc, f1, sup)},
           'confusion': confusion_matrix(y, pred, labels=list(range(n_out))).tolist(),
           'nll': float(-np.mean(np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)))),
           'brier': float(np.mean(np.sum((p - np.eye(n_out)[y]) ** 2, axis=1))),
           'ece_15': ece15(p, y)}
    if threshold is not None:
        acc = p.max(1) >= threshold; n = int(acc.sum())
        out.update({'threshold': float(threshold), 'coverage': float(acc.mean()),
                    'selective_accuracy': float((pred[acc] == y[acc]).mean()) if n else None,
                    'selective_risk': float((pred[acc] != y[acc]).mean()) if n else None,
                    'correct_accepted_yield': float(((pred == y) & acc).mean()),
                    'coverage_per_class': {int(c): float(acc[y == c].mean()) for c in L}})
    return out

def ece15(p, y, bins=15):
    conf = p.max(1); correct = (p.argmax(1) == y).astype(float); e = 0.0
    idx = np.minimum((conf * bins).astype(int), bins - 1)
    for b in range(bins):
        m = idx == b
        if m.any(): e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

def fast_accuracy(y, pred):
    return float(np.mean(y == pred))

def fast_macro_f1(y, pred, labels, n_out):
    """Macro-F1 over `labels` from a bincount confusion matrix; identical to sklearn f1_score(average='macro', zero_division=0)."""
    cm = np.bincount(y * n_out + pred, minlength=n_out * n_out).reshape(n_out, n_out)
    tp = np.diag(cm)[labels].astype(float); fp = cm[:, labels].sum(0) - tp; fn = cm[labels, :].sum(1) - tp
    denom = 2 * tp + fp + fn
    return float(np.mean(np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom > 0)))

def group_bootstrap(stat_fn, groups, n_boot=2000, seed=2026):
    """95% percentile interval of stat_fn(index_array), resampling whole groups with replacement (Efron & Tibshirani, 1994)."""
    groups = np.asarray(groups); uniq, inv = np.unique(groups, return_inverse=True)
    order = np.argsort(inv, kind='stable'); lengths = np.bincount(inv); starts = np.concatenate([[0], np.cumsum(lengths)[:-1]])
    rng = np.random.default_rng(seed); vals = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, len(uniq), len(uniq)); lens = lengths[pick]
        offs = np.repeat(starts[pick] - np.cumsum(lens) + lens, lens) + np.arange(lens.sum())
        vals[b] = stat_fn(order[offs])
    return {'estimate': float(stat_fn(np.arange(len(groups)))), 'ci95': [float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))],
            'n_boot': n_boot, 'n_groups': int(len(uniq))}

def reliability_bins(p, y, bins=15):
    conf = p.max(1); correct = (p.argmax(1) == y).astype(float)
    idx = np.minimum((conf * bins).astype(int), bins - 1); rows = []
    for b in range(bins):
        m = idx == b
        rows.append({'bin_low': b / bins, 'bin_high': (b + 1) / bins, 'n': int(m.sum()),
                     'mean_confidence': float(conf[m].mean()) if m.any() else None, 'accuracy': float(correct[m].mean()) if m.any() else None})
    return rows

def risk_coverage(p, y):
    order = np.argsort(-p.max(1)); correct = (p.argmax(1) == y)[order]
    k = np.arange(1, len(y) + 1)
    return k / len(y), 1 - np.cumsum(correct) / k


# 6. Endpoints with group bootstrap (groups = CCMT leaf groups), tables, zip
INV = {crop: {v: k for k, v in LABEL_IDX[crop].items()} for crop in CLASSES}
res = {'notebook': '08', 'status': 'exploratory (D3 seen in Phase 1); causal design', 'conditions': CONDITIONS, 'crops': {}}
tables = []
for crop in CLASSES:
    P = PRED[PRED.crop == crop]; labels = [LABEL_IDX[crop][c] for c in CLASSES[crop]]; n_out = N_OUT[crop]
    base = P[P.condition == 'original'].set_index('sample_id').sort_index()
    y = base.y.values; groups = base.group_id.values; pred0 = base.pred.values
    maj = float(np.bincount(y, minlength=n_out).max() / len(y))
    out = {'n_leaves': int(len(base)), 'n_groups': int(base.group_id.nunique()), 'majority_class_rate': round(maj, 4), 'by_condition': {}}
    for c in CONDITIONS:
        Q = P[P.condition == c].set_index('sample_id').loc[base.index]; pred = Q.pred.values; conf = Q.conf.values; acc = Q.accept.values
        row = {'accuracy': group_bootstrap(lambda ix: fast_accuracy(y[ix], pred[ix]), groups),
               'macro_f1': group_bootstrap(lambda ix: fast_macro_f1(y[ix], pred[ix], labels, n_out), groups),
               'mean_confidence': float(conf.mean()), 'coverage': float(acc.mean()),
               'recall': {INV[crop][l]: group_bootstrap(lambda ix, l=l: float(np.mean(pred[ix][y[ix] == l] == l)) if (y[ix] == l).any() else np.nan, groups) for l in labels}}
        if c != 'original':
            chg = (pred != pred0).astype(float)
            row['prediction_changed'] = group_bootstrap(lambda ix: float(chg[ix].mean()), groups)
            row['prediction_changed_by_class'] = {INV[crop][l]: float(chg[y == l].mean()) for l in labels}
            row['agreement_with_original_prediction'] = float((pred == pred0).mean())
            for l in labels:        # paired recall difference versus the original photo
                m = y == l; dl = (pred == l).astype(float) - (pred0 == l).astype(float)
                row['recall'][INV[crop][l]]['diff_vs_original'] = group_bootstrap(lambda ix, m=m, dl=dl: float(dl[ix][m[ix]].mean()) if m[ix].any() else np.nan, groups)
        if c == 'field':
            row['by_background_label'] = {}
            for bl, g in Q.groupby('bg_label'):
                yy = base.loc[g.index].y.values; pp = g.pred.values
                row['by_background_label'][bl] = {'n': int(len(g)), 'accuracy': float(np.mean(yy == pp)), 'recall': {INV[crop][l]: float(np.mean(pp[yy == l] == l)) if (yy == l).any() else None for l in labels}}
        if c == 'texture':
            row['by_texture_kind'] = {k: {'n': int(len(g)), 'accuracy': float(np.mean(base.loc[g.index].y.values == g.pred.values))} for k, g in Q.groupby('texture_kind')}
        out['by_condition'][c] = row
        tables.append({'crop': crop, 'condition': c, 'n': len(base), 'accuracy': row['accuracy']['estimate'], 'acc_lo': row['accuracy']['ci95'][0], 'acc_hi': row['accuracy']['ci95'][1],
                       'macro_f1': row['macro_f1']['estimate'], 'changed_vs_original': row.get('prediction_changed', {}).get('estimate'),
                       'changed_lo': row.get('prediction_changed', {}).get('ci95', [None, None])[0], 'changed_hi': row.get('prediction_changed', {}).get('ci95', [None, None])[1],
                       'mean_conf': row['mean_confidence'], 'coverage': row['coverage'],
                       **{f'recall_{INV[crop][l]}': row['recall'][INV[crop][l]]['estimate'] for l in labels}})
    bo = out['by_condition']['background_only']
    out['background_only_vs_majority'] = {'accuracy': bo['accuracy']['estimate'], 'ci95': bo['accuracy']['ci95'], 'majority_class_rate': maj,
                                          'above_majority': bool(bo['accuracy']['ci95'][0] > maj), 'agreement_with_original_prediction': bo['agreement_with_original_prediction']}
    res['crops'][crop] = out
json.dump(res, open(f'{OUT}/e12_endpoints.json', 'w'), indent=1, default=float)
TAB = pd.DataFrame(tables); TAB.to_csv(f'{OUT}/e12_summary.csv', index=False)
pd.set_option('display.width', 200)
for crop in CLASSES:
    print(f'\n=== {crop}: Phase 1 model, {res["crops"][crop]["n_leaves"]} CCMT leaves, majority-class rate {res["crops"][crop]["majority_class_rate"]:.3f} ===')
    cols = ['condition', 'accuracy', 'acc_lo', 'acc_hi', 'macro_f1', 'changed_vs_original', 'changed_lo', 'changed_hi', 'mean_conf', 'coverage'] + [c for c in TAB.columns if c.startswith('recall_') and TAB[TAB.crop == crop][c].notna().any()]
    print(TAB[TAB.crop == crop][cols].round(3).to_string(index=False))
    print('background only vs majority:', json.dumps(res['crops'][crop]['background_only_vs_majority']))
    print('field background, by Ugandan background label:', json.dumps(res['crops'][crop]['by_condition']['field']['by_background_label']))
import zipfile
zp = '/content/nb08_results.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f'{OUT}/*') + glob.glob(f'{SHEETS}/*'):
        if os.path.isfile(p): z.write(p, os.path.relpath(p, OUT))
from google.colab import files
files.download(zp); print('nb08_results.zip ready')

### When it finishes
The run records are written to Google Drive under `ocdf/records/` and also downloaded as `nb08_results.zip`. The copies used for the published results are in this repository under `records/nb08/`.